# C1 · 3. Train and evaluate the stutter-type head

The plan's model: a small head on **wav2vec 2.0 (XLS-R) embeddings** plus **MFCC / pitch / energy** features, trained on SEP-28k with the **speaker-separated SEP-28k-E splits**, reported as **macro-F1** over the five types (block, prolongation, sound repetition, word repetition, interjection).

1. Pick the XLS-R layer on the **dev** split.
2. Compare inputs (acoustic only / XLS-R only / both) and heads (logistic regression / small MLP); per-label thresholds are tuned on dev.
3. Report the chosen model on the untouched **test** split, next to the service's current **rule-based acoustic detector** on the same clips.
4. Export `models/c1_head.joblib`; the service uses it with `C1_DETECTOR=xlsr`.

The dataset plan quotes about **0.45 macro-F1** for published systems on these splits; check the paper's exact setting before comparing.

Settings: `SWARA_C1_MIN_VOTES` (default 2 of 3 annotators), `SWARA_C1_HEADS` (default `logreg,mlp`), `SWARA_C1_HEAD_OUT` (default `models/c1_head.joblib`), `SWARA_C1_REPORT_DIR` (default `training/reports`), `SWARA_C1_NOTE`.

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the speech service's own code (app/, training/)
DATA = Path(os.environ.get("SWARA_C1_DATA", ROOT / "data" / "sep28k"))

MIN_VOTES = int(os.environ.get("SWARA_C1_MIN_VOTES", 2))
HEADS = os.environ.get("SWARA_C1_HEADS", "logreg,mlp").split(",")
HEAD_OUT = Path(os.environ.get("SWARA_C1_HEAD_OUT", ROOT / "models" / "c1_head.joblib"))
NOTE = os.environ.get("SWARA_C1_NOTE", "")
REPORT_DIR = Path(os.environ.get("SWARA_C1_REPORT_DIR", ROOT / "training" / "reports"))
os.environ.setdefault("C1_MODEL_DIR", str(HEAD_OUT.parent))  # the service reads the head from here (cell 7)

from training import heads, sep28k
index = pd.read_csv(DATA / "clips_index.csv", dtype={"EpId": str, "ClipId": str})
fmeta = json.loads((DATA / "features" / "meta.json").read_text())
A = np.load(DATA / "features" / "acoustic.npy")
X = {L: np.load(DATA / "features" / f"xlsr_L{L}.npy") for L in fmeta["layers"]}
y = sep28k.targets(index, MIN_VOTES)
keep = sep28k.usable(index)
split = index["SEP28k-E"].to_numpy()
S = {s: keep & (split == s) for s in ("train", "dev", "test")}
LABELS = sep28k.LABELS
print({s: int(m.sum()) for s, m in S.items()}, "clips;", "positives in train:", dict(zip(LABELS, y[S["train"]].sum(axis=0).tolist())))

## 1. XLS-R layer, chosen on dev

In [ ]:
def fit_eval(Xall, kind="logreg"):
    model = heads.make_head(kind).fit(Xall[S["train"]], y[S["train"]])
    p_dev = heads.proba(model, Xall[S["dev"]])
    th = heads.tune_thresholds(p_dev, y[S["dev"]], LABELS)
    return model, th, heads.evaluate(p_dev, y[S["dev"]], LABELS, th)["macroF1"]

layer_scores = {}
for L, x in X.items():
    layer_scores[L] = fit_eval(x)[2]
    print(f"layer {L:2d}: dev macro-F1 {layer_scores[L]:.3f}")
BEST_LAYER = max(layer_scores, key=layer_scores.get)
print("best layer:", BEST_LAYER)

## 2. Inputs × heads (thresholds tuned on dev)

In [ ]:
INPUTS = {
    "acoustic": (["acoustic"], A),
    f"xlsr L{BEST_LAYER}": (["xlsr"], X[BEST_LAYER]),
    f"xlsr L{BEST_LAYER} + acoustic": (["xlsr", "acoustic"], np.concatenate([X[BEST_LAYER], A], axis=1)),
}
runs = []
for name, (inputs, Xall) in INPUTS.items():
    for kind in HEADS:
        t0 = time.time()
        model, th, dev_f1 = fit_eval(Xall, kind)
        runs.append({"inputs": name, "head": kind, "devMacroF1": dev_f1, "model": model, "thresholds": th,
                     "sources": inputs, "X": Xall, "sec": time.time() - t0})
        print(f"{name:28s} {kind:6s} dev macro-F1 {dev_f1:.3f} ({time.time() - t0:.0f} s)")
best = max(runs, key=lambda r: r["devMacroF1"])
print("chosen on dev:", best["inputs"], best["head"])

## 3. Test split: the chosen head vs the rule-based detector

In [ ]:
test_paths = index.loc[S["test"], "path"].tolist()
p_test = heads.proba(best["model"], best["X"][S["test"]])
result = heads.evaluate(p_test, y[S["test"]], LABELS, best["thresholds"])
t0 = time.time()
rules = heads.evaluate(heads.rule_baseline(test_paths, LABELS), y[S["test"]], LABELS)
print(f"rule baseline on {len(test_paths)} clips: {time.time() - t0:.0f} s")

rows = []
for lab in LABELS:
    rows.append({"label": lab, "positives": result["perLabel"][lab]["positives"],
                 "trained F1": result["perLabel"][lab]["f1"], "precision": result["perLabel"][lab]["precision"],
                 "recall": result["perLabel"][lab]["recall"], "rule-based F1": rules["perLabel"][lab]["f1"]})
rows.append({"label": "macro-F1", "trained F1": result["macroF1"], "rule-based F1": rules["macroF1"]})
table = pd.DataFrame(rows).set_index("label").round(3)
table

In [ ]:
# Every input/head combination on test too, for the report (the choice above used dev only).
all_rows = []
for r in runs:
    pt = heads.proba(r["model"], r["X"][S["test"]])
    all_rows.append({"inputs": r["inputs"], "head": r["head"], "dev macro-F1": r["devMacroF1"],
                     "test macro-F1": heads.evaluate(pt, y[S["test"]], LABELS, r["thresholds"])["macroF1"]})
comparison = pd.DataFrame(all_rows).round(3)
comparison

## 4. Export the head and check that the service uses it

In [ ]:
metrics = {"split": "SEP28k-E", "minVotes": MIN_VOTES, "layer": BEST_LAYER, "chosenOn": "dev macro-F1",
           "test": result, "ruleBaselineTest": rules, "comparison": all_rows, "clips": {s: int(m.sum()) for s, m in S.items()}}
version = heads.export_head(HEAD_OUT, best["model"], best["sources"], LABELS, best["thresholds"],
                            BEST_LAYER if "xlsr" in best["sources"] else None, fmeta["encoder"], metrics, NOTE)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / f"{version}.json").write_text(json.dumps(metrics, indent=1, default=float))
(REPORT_DIR / f"{version}.md").write_text(
    f"# {version}\n\n{NOTE}\n\nChosen on dev: {best['inputs']} / {best['head']}.\n\n## Test split (SEP-28k-E)\n\n"
    + table.to_markdown() + "\n\n## All combinations\n\n" + comparison.to_markdown(index=False) + "\n")
print("exported", HEAD_OUT, version)

In [ ]:
# The service's detector, reading the exported file: it must load and score a recording end to end.
import io, soundfile as sf
from app.config import get_settings
from app.detection.xlsr import XlsrDetector
from app import pipeline

det = XlsrDetector()
print("availability:", det.availability())
os.environ["C1_DETECTOR"] = "xlsr"
get_settings.cache_clear()
sample = np.concatenate([sf.read(p, dtype="float32")[0] for p in test_paths[:4]])
buf = io.BytesIO(); sf.write(buf, sample, 16000, format="WAV")
res = pipeline.analyze(buf.getvalue(), "picture_description", "en", transcript="one two three four five six seven eight nine ten")
print("detector:", res.detector if hasattr(res, "detector") else "?", "| events:", len(res.events), "| %SS:", res.percentSS)